# ECB Shocks x Macaulay Duration


Dieses Notebook nutzt **nur** drei Inputs:
- `intermediate/EQDuration_Macaulay.parquet`
- `intermediate/daily_returns_beta.parquet`
- `tables/shocks_ecb_mpd_me_d.csv`

Ziel: Regression von Aktienreaktionen auf ECB-Schocks mit Interaktion auf Macaulay Duration.


In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
import statsmodels.formula.api as smf
import statsmodels.api as sm

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 140)

BASE_DIR = Path("/Users/jakob/Documents/Parforceleistung/Studium/MSc Economics/Masterarbeit/Project_Data")
DATA_DIR = BASE_DIR / "intermediate"
TABLE_DIR = BASE_DIR / "tables"

DUR_PATH = DATA_DIR / "EQDuration_Macaulay.parquet"
RET_PATH = DATA_DIR / "daily_returns_beta.parquet"
SHK_PATH = TABLE_DIR / "shocks_ecb_mpd_me_d.csv"

for p in [DUR_PATH, RET_PATH, SHK_PATH]:
    if not p.exists():
        raise FileNotFoundError(f"Missing required input: {p}")


def zscore_by_year(df: pd.DataFrame, col: str, year_col: str = "YEAR") -> pd.Series:
    def _z(s: pd.Series) -> pd.Series:
        mu = s.mean(skipna=True)
        sd = s.std(skipna=True, ddof=0)
        if pd.isna(sd) or sd == 0:
            return pd.Series(pd.NA, index=s.index)
        return (s - mu) / sd
    return df.groupby(year_col)[col].transform(_z)


def _cluster_groups(data: pd.DataFrame, date_col: str, firm_col: str) -> pd.DataFrame:
    d = data.copy()
    d[date_col] = pd.to_datetime(d[date_col], errors="coerce")
    if d[date_col].isna().any():
        raise ValueError(f"NaT found in {date_col}")

    d[firm_col] = d[firm_col].astype(str).str.strip()
    if (d[firm_col] == "").any():
        raise ValueError(f"Empty values in {firm_col}")

    return pd.DataFrame(
        {
            "g_date": d[date_col].astype("int64"),
            "g_firm": d[firm_col].astype("category").cat.codes.astype("int64"),
        },
        index=d.index,
    )


def _full_rank_columns(X: pd.DataFrame, tol: float = 1e-12):
    cols = list(X.columns)
    if len(cols) <= 1:
        return cols

    keep = cols.copy()
    while len(keep) > 1:
        mat = X[keep].to_numpy(dtype=float)
        rank = np.linalg.matrix_rank(mat, tol=tol)
        if rank == len(keep):
            break
        # drop weakest signal column among current set
        variances = X[keep].var(axis=0, skipna=True).fillna(0.0)
        drop_col = variances.idxmin()
        keep.remove(drop_col)
    return keep


def fit_event_fe_2way(
    data: pd.DataFrame,
    y_col: str,
    x_cols: list,
    date_col: str = "date",
    firm_col: str = "firm_id",
    weights=None,
):
    cols = [y_col, date_col, firm_col] + x_cols
    d = data[cols].dropna().copy()
    d[date_col] = pd.to_datetime(d[date_col], errors="coerce")

    if d.empty:
        raise ValueError("Empty regression sample after dropna.")

    # Event fixed effects via within transformation
    dm_cols = []
    for c in [y_col] + x_cols:
        c_dm = f"{c}__dm"
        d[c_dm] = d[c] - d.groupby(date_col)[c].transform("mean")
        dm_cols.append(c_dm)

    y_dm = f"{y_col}__dm"
    x_dm = [f"{c}__dm" for c in x_cols]

    # remove regressors with (near) zero variance after demeaning
    nonzero = []
    for c in x_dm:
        v = pd.to_numeric(d[c], errors="coerce").var(skipna=True)
        if pd.notna(v) and v > 1e-14:
            nonzero.append(c)

    if not nonzero:
        raise ValueError("No regressor variance left after event demeaning.")

    X = d[nonzero].astype(float)
    keep = _full_rank_columns(X)
    X = X[keep]
    y = d[y_dm].astype(float)

    groups = _cluster_groups(d, date_col=date_col, firm_col=firm_col)

    if weights is None:
        m = sm.OLS(y, X).fit(
            cov_type="cluster",
            cov_kwds={"groups": groups, "use_correction": True},
        )
    else:
        w = pd.Series(weights, index=data.index).reindex(d.index).astype(float)
        m = sm.WLS(y, X, weights=w).fit(
            cov_type="cluster",
            cov_kwds={"groups": groups, "use_correction": True},
        )

    # map back de-meaned regressor names to original labels
    name_map = {f"{c}__dm": c for c in x_cols}
    m.params.index = [name_map.get(i, i) for i in m.params.index]
    m.bse.index = [name_map.get(i, i) for i in m.bse.index]
    m.tvalues.index = [name_map.get(i, i) for i in m.tvalues.index]
    m.pvalues.index = [name_map.get(i, i) for i in m.pvalues.index]

    return m, d, keep


In [2]:
# 1) Load and clean shock data

df_shk = pd.read_csv(SHK_PATH).copy()
df_shk["date"] = pd.to_datetime(df_shk["date"], errors="coerce")
assert df_shk["date"].notna().all(), "Some shock dates could not be parsed."

SHOCK_VERSION = "median"  # alternative: "pm"
if SHOCK_VERSION == "median":
    shock_map = {"MP_median": "ShockMP", "CBI_median": "ShockInfo"}
elif SHOCK_VERSION == "pm":
    shock_map = {"MP_pm": "ShockMP", "CBI_pm": "ShockInfo"}
else:
    raise ValueError("SHOCK_VERSION must be 'median' or 'pm'.")

missing = [c for c in shock_map if c not in df_shk.columns]
if missing:
    raise ValueError(f"Missing shock columns: {missing}")

df_shk = df_shk.rename(columns=shock_map)
for c in ["ShockMP", "ShockInfo"]:
    df_shk[c] = pd.to_numeric(df_shk[c], errors="coerce")

df_shk = (
    df_shk[["date", "ShockMP", "ShockInfo"]]
    .dropna(subset=["date"])
    .drop_duplicates(subset=["date"])
    .sort_values("date")
    .reset_index(drop=True)
)

print("Shock sample:", df_shk.shape)
display(df_shk.head())


Shock sample: (312, 3)


,date,ShockMP,ShockInfo
0,1999-01-07,0.020578,-0.058123
1,1999-01-21,0.008569,-0.004988
2,1999-02-18,-0.005565,0.005565
3,1999-03-04,-0.003596,0.001670
4,1999-03-18,-0.002326,0.001568


In [3]:
# 2) Load and clean Macaulay duration panel

df_dur = pd.read_parquet(DUR_PATH).copy()

if "status" in df_dur.columns:
    df_dur = df_dur[df_dur["status"].eq("ok")].copy()

if "RIC" not in df_dur.columns:
    raise ValueError("RIC missing in EQDuration_Macaulay.parquet")
if "Duration_DCF_Macaulay" not in df_dur.columns:
    raise ValueError("Duration_DCF_Macaulay missing in EQDuration_Macaulay.parquet")

if "YEAR" in df_dur.columns:
    pass
elif "year" in df_dur.columns:
    df_dur = df_dur.rename(columns={"year": "YEAR"})
elif "asof_date" in df_dur.columns:
    df_dur["YEAR"] = pd.to_datetime(df_dur["asof_date"], errors="coerce").dt.year
elif "date" in df_dur.columns:
    df_dur["YEAR"] = pd.to_datetime(df_dur["date"], errors="coerce").dt.year
else:
    raise ValueError("Could not derive YEAR in EQDuration_Macaulay.parquet")

df_dur["RIC"] = df_dur["RIC"].astype(str).str.strip()
df_dur["YEAR"] = pd.to_numeric(df_dur["YEAR"], errors="coerce").astype("Int64")
df_dur["Duration_DCF_Macaulay"] = pd.to_numeric(df_dur["Duration_DCF_Macaulay"], errors="coerce")

df_dur_y = (
    df_dur[["RIC", "YEAR", "Duration_DCF_Macaulay"]]
    .dropna(subset=["RIC", "YEAR", "Duration_DCF_Macaulay"])
    .groupby(["RIC", "YEAR"], as_index=False)["Duration_DCF_Macaulay"]
    .median()
)

print("Duration firm-year sample:", df_dur_y.shape)
display(df_dur_y.head())


Duration firm-year sample: (9107, 3)


,RIC,YEAR,Duration_DCF_Macaulay
0,1910.HK,2011,15.118739
1,1910.HK,2012,14.657776
2,1910.HK,2013,14.684711
3,1910.HK,2014,14.504502
4,1910.HK,2015,14.468720


In [4]:
# 3) Load and clean daily returns panel (AR + beta source)

df_ret = pd.read_parquet(RET_PATH).copy()

if "RIC" not in df_ret.columns or "date" not in df_ret.columns:
    raise ValueError("daily_returns_beta requires at least RIC and date")

df_ret["RIC"] = df_ret["RIC"].astype(str).str.strip()
df_ret["date"] = pd.to_datetime(df_ret["date"], errors="coerce")
df_ret = df_ret.dropna(subset=["RIC", "date"]).copy()

if "firm_id" in df_ret.columns:
    df_ret["firm_id"] = df_ret["firm_id"].astype(str).str.strip()
    df_ret.loc[df_ret["firm_id"] == "", "firm_id"] = pd.NA
    df_ret["firm_id"] = df_ret["firm_id"].fillna(df_ret["RIC"])
else:
    df_ret["firm_id"] = df_ret["RIC"]

# AR is always computed using market_ret_cap80 from daily_returns_beta
MKT_COL = "market_ret_cap80"
if MKT_COL not in df_ret.columns:
    raise ValueError("market_ret_cap80 missing in daily_returns_beta.parquet")

# Firm return column (exclude market return and metadata return-like fields)
ret_priority = [
    "ret", "return", "returns", "daily_return", "stock_ret", "ret_daily",
    "tr.totalreturn", "tr.totalreturn1d", "tr.totalreturngross", "tr.pricepctchg", "pct_change", "r"
]
ret_candidates = [
    c for c in ret_priority
    if c in df_ret.columns and c != MKT_COL
]

if not ret_candidates:
    ret_candidates = [
        c for c in df_ret.columns
        if (
            ("ret" in c.lower() or "return" in c.lower())
            and c not in {MKT_COL, "AR", "abret", "abnormal_return", "beta_capm_daily"}
            and not c.lower().startswith("market_")
        )
    ]

if not ret_candidates:
    raise ValueError("No firm return column found in daily_returns_beta.parquet to compute AR")

RET_COL = ret_candidates[0]

df_ret[RET_COL] = pd.to_numeric(df_ret[RET_COL], errors="coerce")
df_ret[MKT_COL] = pd.to_numeric(df_ret[MKT_COL], errors="coerce")
df_ret["AR"] = df_ret[RET_COL] - df_ret[MKT_COL]

# Beta source in daily_returns_beta file
BETA_COL = "beta_capm_daily"
if BETA_COL not in df_ret.columns:
    raise ValueError("beta_capm_daily missing in daily_returns_beta.parquet")

df_ret[BETA_COL] = pd.to_numeric(df_ret[BETA_COL], errors="coerce")

print("Returns sample:", df_ret.shape)
print("Using return column for AR:", RET_COL)
print("Using market column for AR:", MKT_COL)
print("Using beta column:", BETA_COL)
display(df_ret[["firm_id", "RIC", "date", RET_COL, MKT_COL, "AR", BETA_COL]].head())



Returns sample: (3336240, 11)
Using return column for AR: ret
Using market column for AR: market_ret_cap80
Using beta column: beta_capm_daily


,firm_id,RIC,date,ret,market_ret_cap80,AR,beta_capm_daily
0,FIRM0000002,STRV.VI,2008-01-02,0.009647,-0.011095,0.020742,NaN
1,FIRM0000002,STRV.VI,2008-01-03,0.021143,-0.003918,0.02506,NaN
2,FIRM0000002,STRV.VI,2008-01-04,-0.01951,-0.016911,-0.002599,NaN
3,FIRM0000002,STRV.VI,2008-01-07,-0.045279,-0.002236,-0.043043,NaN
4,FIRM0000002,STRV.VI,2008-01-08,0.007018,0.007045,-0.000026,NaN


In [5]:
# 4) Build event panel and merge shocks + predetermined duration + predetermined beta

df_evt = df_ret[df_ret["date"].isin(df_shk["date"])].copy()

df_evt = df_evt.merge(
    df_shk[["date", "ShockMP", "ShockInfo"]],
    on="date",
    how="left",
    validate="m:1",
)

# Predetermined year t-1
df_evt["YEAR"] = (df_evt["date"].dt.year - 1).astype("Int64")

# Merge duration (RIC x YEAR)
df_evt = df_evt.merge(
    df_dur_y,
    on=["RIC", "YEAR"],
    how="left",
    validate="m:1",
)

# Build firm-year beta from daily returns, then merge predetermined beta
beta_fy = (
    df_ret.assign(YEAR=pd.to_datetime(df_ret["date"]).dt.year.astype("Int64"))
    [["RIC", "YEAR", BETA_COL]]
    .dropna(subset=["RIC", "YEAR", BETA_COL])
    .groupby(["RIC", "YEAR"], as_index=False)[BETA_COL]
    .median()
    .rename(columns={BETA_COL: "BETA_5Y"})
)

df_evt = df_evt.merge(
    beta_fy,
    on=["RIC", "YEAR"],
    how="left",
    validate="m:1",
)

# YEAR-wise standardization
for col in ["Duration_DCF_Macaulay", "BETA_5Y"]:
    df_evt[f"{col}_std"] = zscore_by_year(df_evt, col, year_col="YEAR")

print("Event panel:", df_evt.shape)
print("Unique dates:", df_evt["date"].nunique(), "| Unique firms:", df_evt["firm_id"].nunique())
display(df_evt[["firm_id", "RIC", "date", "YEAR", "AR", "ShockMP", "ShockInfo", "Duration_DCF_Macaulay", "Duration_DCF_Macaulay_std", "BETA_5Y", "BETA_5Y_std"]].head(10))


Event panel: (149565, 18)
Unique dates: 312 | Unique firms: 1213


,firm_id,RIC,date,YEAR,AR,ShockMP,ShockInfo,Duration_DCF_Macaulay,Duration_DCF_Macaulay_std,BETA_5Y,BETA_5Y_std
0,FIRM0000002,STRV.VI,2008-01-10,2007,0.009223,0.008544,-0.010348,NaN,NaN,NaN,NaN
1,FIRM0000002,STRV.VI,2008-02-07,2007,0.004053,-0.005656,-0.044323,NaN,NaN,NaN,NaN
2,FIRM0000002,STRV.VI,2008-03-06,2007,0.013678,0.032879,-0.003196,NaN,NaN,NaN,NaN
3,FIRM0000002,STRV.VI,2008-04-10,2007,-0.001876,-0.001077,0.002117,NaN,NaN,NaN,NaN
4,FIRM0000002,STRV.VI,2008-05-08,2007,-0.004307,0.007965,-0.019217,NaN,NaN,NaN,NaN
5,FIRM0000002,STRV.VI,2008-06-05,2007,0.00072,0.087185,0.059450,NaN,NaN,NaN,NaN
6,FIRM0000002,STRV.VI,2008-07-03,2007,-0.023505,-0.086166,0.000316,NaN,NaN,NaN,NaN
7,FIRM0000002,STRV.VI,2008-08-07,2007,-0.011566,-0.000476,-0.041203,NaN,NaN,NaN,NaN
8,FIRM0000002,STRV.VI,2008-09-04,2007,-0.016194,0.020624,-0.027658,NaN,NaN,NaN,NaN
9,FIRM0000002,STRV.VI,2008-10-02,2007,-0.048175,-0.002948,-0.054763,NaN,NaN,NaN,NaN


## Baseline Regression

Spezifikation mit Event-FE und 2-way Clustering auf `date` und `firm_id`.


In [6]:
BASE_DUR_STD = "Duration_DCF_Macaulay_std"

reg_cols = ["AR", "ShockMP", "ShockInfo", "date", "firm_id", BASE_DUR_STD, "BETA_5Y_std"]
df_reg = df_evt[reg_cols].dropna().copy()

x_cols = [
    f"ShockMP:{BASE_DUR_STD}",
    f"ShockInfo:{BASE_DUR_STD}",
    "ShockMP:BETA_5Y_std",
    "ShockInfo:BETA_5Y_std",
]

# build explicit interaction columns
for c in x_cols:
    a, b = c.split(":")
    df_reg[c] = pd.to_numeric(df_reg[a], errors="coerce") * pd.to_numeric(df_reg[b], errors="coerce")

m_base, d_base, keep_base = fit_event_fe_2way(
    data=df_reg,
    y_col="AR",
    x_cols=x_cols,
    date_col="date",
    firm_col="firm_id",
)

res_base = pd.DataFrame({
    "coef": m_base.params,
    "std_err": m_base.bse,
    "t": m_base.tvalues,
    "pvalue": m_base.pvalues,
})

print("Baseline sample:", d_base.shape)
print("Regressors kept:", [k.replace("__dm", "") for k in keep_base])
display(res_base)


Baseline sample: (85436, 12)
Regressors kept: ['ShockMP:Duration_DCF_Macaulay_std', 'ShockInfo:Duration_DCF_Macaulay_std', 'ShockMP:BETA_5Y_std', 'ShockInfo:BETA_5Y_std']


,coef,std_err,t,pvalue
ShockMP:Duration_DCF_Macaulay_std__dm,-0.001509,0.003326,-0.453664,6.500704e-01
ShockInfo:Duration_DCF_Macaulay_std__dm,0.010736,0.003120,3.441415,5.786808e-04
ShockMP:BETA_5Y_std__dm,-0.033188,0.011784,-2.816271,4.858462e-03
ShockInfo:BETA_5Y_std__dm,0.059643,0.012169,4.901174,9.526539e-07


## Robustness 1: Event Equal Weights

Jedes Event bekommt gleiches Gesamtgewicht.


In [7]:
df_reg_w = df_reg.copy()
df_reg_w["w_event_equal"] = 1.0 / df_reg_w.groupby("date")["date"].transform("size")
df_reg_w["w_event_equal"] = df_reg_w["w_event_equal"] / df_reg_w["w_event_equal"].mean()

m_w, d_w, keep_w = fit_event_fe_2way(
    data=df_reg_w,
    y_col="AR",
    x_cols=x_cols,
    date_col="date",
    firm_col="firm_id",
    weights=df_reg_w["w_event_equal"],
)

res_w = pd.DataFrame({
    "coef_w": m_w.params,
    "std_err_w": m_w.bse,
    "t_w": m_w.tvalues,
    "pvalue_w": m_w.pvalues,
})

cmp = res_base.join(res_w, how="outer")
print("Event-weighted sample:", d_w.shape)
print("Regressors kept:", [k.replace("__dm", "") for k in keep_w])
display(cmp)


Event-weighted sample: (85436, 12)
Regressors kept: ['ShockMP:Duration_DCF_Macaulay_std', 'ShockInfo:Duration_DCF_Macaulay_std', 'ShockMP:BETA_5Y_std', 'ShockInfo:BETA_5Y_std']


,coef,std_err,t,pvalue,coef_w,std_err_w,t_w,pvalue_w
ShockInfo:BETA_5Y_std__dm,0.059643,0.012169,4.901174,9.526539e-07,0.061556,0.012053,5.107004,3.273067e-07
ShockInfo:Duration_DCF_Macaulay_std__dm,0.010736,0.003120,3.441415,5.786808e-04,0.012657,0.003221,3.928911,8.533132e-05
ShockMP:BETA_5Y_std__dm,-0.033188,0.011784,-2.816271,4.858462e-03,-0.035745,0.011657,-3.066475,2.165992e-03
ShockMP:Duration_DCF_Macaulay_std__dm,-0.001509,0.003326,-0.453664,6.500704e-01,-0.002643,0.003356,-0.787588,4.309379e-01


## Robustness 2: AR[0,+1]


In [8]:
df_ret_01 = df_ret.sort_values(["firm_id", "date"]).copy()
df_ret_01["AR0"] = pd.to_numeric(df_ret_01["AR"], errors="coerce")
df_ret_01["AR1"] = df_ret_01.groupby("firm_id")["AR0"].shift(-1)
df_ret_01["AR_01"] = df_ret_01["AR0"] + df_ret_01["AR1"]

df_evt_01 = df_ret_01[df_ret_01["date"].isin(df_shk["date"])].copy()
df_evt_01 = df_evt_01.merge(df_shk, on="date", how="left", validate="m:1")
df_evt_01["YEAR"] = (df_evt_01["date"].dt.year - 1).astype("Int64")

# merge duration + predetermined beta
df_evt_01 = df_evt_01.merge(df_dur_y, on=["RIC", "YEAR"], how="left", validate="m:1")
df_evt_01 = df_evt_01.merge(beta_fy, on=["RIC", "YEAR"], how="left", validate="m:1")

for col in ["Duration_DCF_Macaulay", "BETA_5Y"]:
    df_evt_01[f"{col}_std"] = zscore_by_year(df_evt_01, col, year_col="YEAR")

reg_cols_01 = ["AR_01", "ShockMP", "ShockInfo", "date", "firm_id", "Duration_DCF_Macaulay_std", "BETA_5Y_std"]
df_reg_01 = df_evt_01[reg_cols_01].dropna().copy()

x_cols_01 = [
    "ShockMP:Duration_DCF_Macaulay_std",
    "ShockInfo:Duration_DCF_Macaulay_std",
    "ShockMP:BETA_5Y_std",
    "ShockInfo:BETA_5Y_std",
]
for c in x_cols_01:
    a, b = c.split(":")
    df_reg_01[c] = pd.to_numeric(df_reg_01[a], errors="coerce") * pd.to_numeric(df_reg_01[b], errors="coerce")

m_01, d_01, keep_01 = fit_event_fe_2way(
    data=df_reg_01,
    y_col="AR_01",
    x_cols=x_cols_01,
    date_col="date",
    firm_col="firm_id",
)

res_01 = pd.DataFrame({
    "coef": m_01.params,
    "std_err": m_01.bse,
    "t": m_01.tvalues,
    "pvalue": m_01.pvalues,
})

print("AR[0,+1] sample:", d_01.shape)
print("Regressors kept:", [k.replace("__dm", "") for k in keep_01])
display(res_01)


AR[0,+1] sample: (84955, 12)
Regressors kept: ['ShockMP:Duration_DCF_Macaulay_std', 'ShockInfo:Duration_DCF_Macaulay_std', 'ShockMP:BETA_5Y_std', 'ShockInfo:BETA_5Y_std']


,coef,std_err,t,pvalue
ShockMP:Duration_DCF_Macaulay_std__dm,-0.007221,0.005659,-1.276076,0.201929
ShockInfo:Duration_DCF_Macaulay_std__dm,0.009626,0.005158,1.866263,0.062005
ShockMP:BETA_5Y_std__dm,-0.031511,0.012847,-2.452733,0.014178
ShockInfo:BETA_5Y_std__dm,0.061251,0.015212,4.026430,0.000057


## Robustness 3: Portfolio Split (Q1 vs Q5)


In [9]:
df_ps = df_evt.copy()

# quintiles by YEAR

def qbin(s: pd.Series):
    x = pd.to_numeric(s, errors="coerce")
    ok = x.notna()
    if ok.sum() < 50:
        return pd.Series(pd.NA, index=s.index)
    ranks = x[ok].rank(method="average")
    b = pd.qcut(ranks, q=5, labels=["Q1", "Q2", "Q3", "Q4", "Q5"])
    out = pd.Series(pd.NA, index=s.index, dtype="object")
    out.loc[b.index] = b.astype("object")
    return out

df_ps["Dur_bin"] = df_ps.groupby("YEAR")["Duration_DCF_Macaulay"].transform(qbin)
df_ps = df_ps[df_ps["Dur_bin"].isin(["Q1", "Q5"])].copy()
df_ps["HighDur"] = (df_ps["Dur_bin"] == "Q5").astype(int)

reg_cols_ps = ["AR", "ShockMP", "ShockInfo", "date", "firm_id", "HighDur", "BETA_5Y_std"]
df_reg_ps = df_ps[reg_cols_ps].dropna().copy()

x_cols_ps = [
    "ShockMP:HighDur",
    "ShockInfo:HighDur",
    "ShockMP:BETA_5Y_std",
    "ShockInfo:BETA_5Y_std",
]
for c in x_cols_ps:
    a, b = c.split(":")
    df_reg_ps[c] = pd.to_numeric(df_reg_ps[a], errors="coerce") * pd.to_numeric(df_reg_ps[b], errors="coerce")

m_ps, d_ps, keep_ps = fit_event_fe_2way(
    data=df_reg_ps,
    y_col="AR",
    x_cols=x_cols_ps,
    date_col="date",
    firm_col="firm_id",
)

res_ps = pd.DataFrame({
    "coef": m_ps.params,
    "std_err": m_ps.bse,
    "t": m_ps.tvalues,
    "pvalue": m_ps.pvalues,
})

print("Portfolio split sample:", d_ps.shape)
print("Regressors kept:", [k.replace("__dm", "") for k in keep_ps])
display(res_ps)


Portfolio split sample: (34041, 12)
Regressors kept: ['ShockMP:HighDur', 'ShockInfo:HighDur', 'ShockMP:BETA_5Y_std', 'ShockInfo:BETA_5Y_std']


,coef,std_err,t,pvalue
ShockMP:HighDur__dm,-0.006470,0.012029,-0.537909,0.590640
ShockInfo:HighDur__dm,0.028583,0.013661,2.092303,0.036411
ShockMP:BETA_5Y_std__dm,-0.035745,0.012608,-2.834985,0.004583
ShockInfo:BETA_5Y_std__dm,0.059906,0.013886,4.313970,0.000016
